# Classification of Consumer Complaints

The Consumer Financial Protection Bureau publishes the Consumer Complaint Database, a collection of complaints about consumer financial products and services that were sent to companies for response. Complaints are published after the company responds, confirming a commercial relationship with the consumer, or after 15 days, whichever comes first. 

You have been provided with a dataset of over 350,000 such complaints for 5 common issue types. Your goal is to train a text classification model to identify the issue type based on the consumer complaint narrative. The data can be downloaded from https://drive.google.com/file/d/1Hz1gnCCr-SDGjnKgcPbg7Nd3NztOLdxw/view?usp=share_link 

In [5]:
import pandas as pd
import numpy as np

from joblib import dump, load

from sklearn.naive_bayes import MultinomialNB

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix

### open dataset

In [6]:
reviews = pd.read_csv('data/complaints.csv')
reviews.head()

,Consumer complaint narrative,Issue
0,My name is XXXX XXXX this complaint is not mad...,Incorrect information on your report
1,I searched on XXXX for XXXXXXXX XXXX and was ...,Fraud or scam
2,I have a particular account that is stating th...,Incorrect information on your report
3,I have not supplied proof under the doctrine o...,Attempts to collect debt not owed
4,Hello i'm writing regarding account on my cred...,Incorrect information on your report


In [11]:
reviews.groupby('Issue')['Issue'].count()

Issue
Attempts to collect debt not owed        73163
Communication tactics                    21243
Fraud or scam                            12347
Incorrect information on your report    229305
Struggling to pay mortgage               17374
Name: Issue, dtype: int64

In [19]:
#convert all reviews to strings and make it lowercase
reviews['Consumer complaint narrative'] = reviews['Consumer complaint narrative'].astype('str').str.lower()


---

### apply classification models

In [26]:
print(X_train_vec)

  (0, 1)	1
  (0, 0)	1
  (0, 2)	1


In [27]:
predict_var= 'Consumer complaint narrative'
target_var= 'Issue'
X = reviews[[predict_var]]
y = reviews[target_var]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 321, stratify = y)

#vectorize
vect = CountVectorizer()

X_train_vec = vect.fit_transform(X_train[predict_var])
X_test_vec = vect.transform(X_test[predict_var])

#Fit model 
nb = MultinomialNB().fit(X_train_vec, y_train)

y_pred = nb.predict(X_test_vec)

In [32]:
from sklearn.metrics import classification_report

In [34]:
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7988976663120487
[[12086  2039   500  3343   323]
 [  587  4476    51   112    85]
 [   67    55  2813   110    42]
 [ 6610  1046   834 46997  1839]
 [   36    40    12    38  4217]]
                                      precision    recall  f1-score   support

   Attempts to collect debt not owed       0.62      0.66      0.64     18291
               Communication tactics       0.58      0.84      0.69      5311
                       Fraud or scam       0.67      0.91      0.77      3087
Incorrect information on your report       0.93      0.82      0.87     57326
          Struggling to pay mortgage       0.65      0.97      0.78      4343

                            accuracy                           0.80     88358
                           macro avg       0.69      0.84      0.75     88358
                        weighted avg       0.82      0.80      0.80     88358



**observation** Using a monogram model produced an accuracy score of 0.80, but classified the 5 'Issue' classes at differnt F1-score rates (ranging from 0.64 to 0.87)

### Model using Bigrams and Trigrams

In [35]:
vect = CountVectorizer()
clf = MultinomialNB()

pipe = Pipeline([("vect", vect), ("clf", clf)])

param_grid = {
    'vect__ngram_range':[(1,1), (1,2), (1,3)],
    'vect__min_df':[1, 2, 5, 10, 20],
    'clf__fit_prior':[False, True]
}

In [37]:
rs = RandomizedSearchCV(estimator = pipe, param_distributions = param_grid, verbose = 2, n_jobs = -1)
rs.fit(X_train[predict_var], y_train)

print(rs.best_params_)
print(rs.best_score_)

# dump(rs, "../models/cv_01.joblib")

# rs = load("../models/cv_01.joblib")



Fitting 5 folds for each of 10 candidates, totalling 50 fits
{'vect__ngram_range': (1, 2), 'vect__min_df': 1, 'clf__fit_prior': True}
0.8441114500668716


**observation**: The best prediciton model was using the Bigram and clf_fit_prior

In [38]:
y_pred = rs.best_estimator_.predict(X_test[predict_var])

print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8436813870843614
[[14064   714   145  3097   271]
 [ 1055  3994    25   111   126]
 [  126    27  2732   133    69]
 [ 5953   137   188 49515  1533]
 [   38    13     6    45  4241]]
                                      precision    recall  f1-score   support

   Attempts to collect debt not owed       0.66      0.77      0.71     18291
               Communication tactics       0.82      0.75      0.78      5311
                       Fraud or scam       0.88      0.89      0.88      3087
Incorrect information on your report       0.94      0.86      0.90     57326
          Struggling to pay mortgage       0.68      0.98      0.80      4343

                            accuracy                           0.84     88358
                           macro avg       0.80      0.85      0.82     88358
                        weighted avg       0.86      0.84      0.85     88358



**observation** Using a bigram and trigram did improve model accuracy up to 0.84, and imporved all the 5 classification 'Issues' ranging with an f1-score from 0.71 to 0.90. However, the model fitting took a very long time to process.

---

---

As you work, answer the following questions: 
* What steps did you take to preprocess the data?
* How did a model using unigrams compare to one using bigrams or trigrams?
* How did a count vectorizer compare to a tfidf vectorizer?
* What models did you try and how successful were they? Where did they struggle? Were there issues that the models commonly mixed up?
* What words or phrases were most influential on your models' predictions?

**Bonus:** A larger dataset containing 20 additional categories can be downloaded from https://drive.google.com/file/d/1gW6LScUL-Z7mH6gUZn-1aNzm4p4CvtpL/view?usp=share_link. How well do your models work with these additional categories?